# Week 7B — Can an AI improve our land-cover classifier?
**GGS 662 Agentic GeoAI | A simple case study building on GGS416**

**Research question: Which method best predicts pixel land cover from satellite imagery?**

Start with a basic classifier that we understand. Give it real satellite
measurements and reference labels, check its predictions, then ask a coding
agent to investigate whether a different method or preprocessing choice helps.
The goal is to understand the research process, not build an agent framework.

Everything needed for the lesson is in this notebook. The first data cell
creates `week7/landcover/data/` and downloads the training observations and
original documentation from UCI. Later runs reuse these local files. Internet
access is needed for the first download; downloaded data are ignored by Git.
No separate Python modules or test files are required.

We build on your GGS416
[Week 5 Image Classification](https://github.com/edwardoughton/satellite-image-analysis/blob/1cbe05d928445d31cdc424c03345a412436285af/05_01_ggs416_26_02_23.ipynb)
lesson: pixels become feature vectors, a classifier learns from labelled
examples, and a confusion matrix helps us understand mistakes.

## 1. What are we actually trying to predict?

### Why are we building a classifier?

Imagine trying to produce a land-cover map for an entire country—or the world.
A person could inspect imagery and label some pixels, but manually identifying
**every pixel** would be prohibitively time-consuming. Instead, we collect a
manageable set of **reference-labelled examples** and use them to train a
model that predicts the land-cover class of other pixels automatically.

Here is the basic workflow, with terms explained for newcomers:

1. **Reference labels (our best available “truth”):** a known class for each
   sampled central pixel, established using independent evidence. Labels are
   useful but can contain mistakes.
2. **Training:** show a model the satellite measurements **and** labels for
   some examples so it learns patterns linking the two.
3. **Validation:** ask the model to predict labels for different, held-out
   examples whose answers we know. Compare predictions against those answers
   to choose or improve the method.
4. **Final test:** after choosing a method, evaluate it once on another
   untouched set of labelled examples to estimate performance more fairly.
5. **Prediction at scale:** in principle, apply the trained model to new
   satellite pixels without manually labelling each one. **But** reliable
   mapping elsewhere requires suitable imagery, representative training
   examples, and independent tests in the new places and conditions. A high
   score here does **not** prove the model will work worldwide.

**What this particular dataset is—and is not:** We do **not** download a
complete, fully labelled Landsat image. We use 6,435 labelled samples of
**3 × 3 pixel neighbourhoods** (4,435 development samples and 2,000 final
publisher test samples), each with four spectral bands per pixel and a
reference label for the **central** pixel. The source data do not provide
coordinates or identify a usable scene footprint, so **we cannot say where
these samples are located, reconstruct a map, or test transfer to another
region**. UCI (University of California, Irvine) distributes this benchmark.

A satellite sensor measures energy in different wavelength bands. Water,
vegetation and soil often have different spectral responses, but shadows,
mixed pixels and similar materials can make them difficult to distinguish.
The sensor does not directly measure a land-cover class.

For each pixel we need two things:

- **Inputs X:** measured values in several spectral bands.
- **Target y:** a reference class assigned from other evidence, such as a field visit.

A supervised classifier learns a mapping **f(X) → y** from labelled examples.
We then ask how accurately it predicts labels for observations it did not fit.
This differs from segmentation, which groups pixels without necessarily
assigning them a land-cover meaning.

**Think first:** If two classes have similar spectral values, what could help?
More bands? Neighbouring pixels? A different model? Better reference labels?
Each suggestion is a hypothesis we can investigate.

## 2. Use real measurements and reference labels

Your GGS416 example used NAIP aerial imagery and generated provisional labels
from NDVI, NDWI and brightness thresholds. That is a useful way to demonstrate
the steps, but a model could score well simply by learning those same rules.
For this case study we use **Landsat satellite measurements with field-based
reference labels**, rather than generating answers from the input features.

The [UCI Landsat dataset](https://archive.ics.uci.edu/dataset/146/statlog+landsat+satellite)
provides 4,435 development observations. Each row describes a 3 by 3 pixel
neighbourhood: four bands for each of nine pixels, followed by the central
pixel's reference class. Columns 17–20 contain the central pixel's four bands.
The source documentation attributes the classification to a site visit.
Reference labels can still be imperfect.

| Code | Reference class |
| --- | --- |
| 1 | Red soil |
| 2 | Cotton crop |
| 3 | Grey soil |
| 4 | Damp grey soil |
| 5 | Soil with vegetation stubble |
| 7 | Very damp grey soil |

Class 6 is absent. Values are digital numbers, not calibrated surface
reflectance. This is a small soil/crop case study, not every possible land-cover
class. Geographic coordinates and the original scene footprint are not supplied, so
we cannot locate the samples, map the rows, or test geographic separation. Keep that limitation in mind when interpreting scores.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
CLASSES = [1, 2, 3, 4, 5, 7]
CLASS_NAMES = ['Red soil', 'Cotton crop', 'Grey soil', 'Damp grey soil',
               'Vegetation stubble', 'Very damp grey soil']
DATA_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/satimage/'

# Find the notebook's folder, including when the kernel starts in a subfolder.
ROOT = next((folder for folder in [Path.cwd(), *Path.cwd().parents]
             if (folder / '07_02_agentic_land_cover_classification.ipynb').exists()),
            Path.cwd())
DATA_DIR = ROOT / 'week7' / 'landcover' / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)


def download_file(filename):
    path = DATA_DIR / filename
    if path.exists() and path.stat().st_size > 0:
        print('Using local file:', path)
    else:
        print('Downloading:', filename)
        try:
            with urlopen(DATA_URL + filename, timeout=30) as response:
                content = response.read()
        except OSError as error:
            raise RuntimeError(
                f'Could not download {filename}. Check your internet connection '
                'and rerun this cell.') from error
        if not content:
            raise RuntimeError(f'The download of {filename} was empty. Rerun this cell.')
        path.write_bytes(content)
    return path


# Only development data and documentation are downloaded at this stage.
data_path = download_file('sat.trn')
documentation_path = download_file('sat.doc')
data = np.loadtxt(data_path)

X = data[:, :36]             # Measured bands, excluding the reference label.
y = data[:, 36].astype(int)  # Class of the central pixel.
print('Observations:', len(y), 'Measurements per observation:', X.shape[1])
display(pd.DataFrame(X[:5, 16:20], columns=['Band 1', 'Band 2', 'Band 3', 'Band 4']))
display(pd.Series(y).value_counts().sort_index().rename('Reference class counts'))

## 3. Decide what would count as a better answer

Do not evaluate a model on the observations used to fit it. Hold back a fixed
25% of the development observations for **validation**. The AI can use those
validation results to choose its next experiment, but the observations and
labels stay the same for every comparison.

### Reading the performance scores (no prior machine learning required)

We know the **reference label** for every validation sample, so we can compare
what the classifier **predicted** with the answer provided in the dataset.

- **Accuracy:** the fraction of all validation samples classified correctly.
  For example, 80 correct predictions out of 100 gives **80% accuracy**.
  Caution: a model can achieve high accuracy by favoring a common class while
  performing poorly on rare classes.
- **Precision (for one class):** among samples the model **predicted as that
  class**, what fraction really belonged to it? High precision means few
  false alarms for that class.
- **Recall (for one class):** among samples that **really belong to that
  class**, what fraction did the model find? High recall means few misses.
- **F1 score (for one class):** a single number balancing precision and recall:
  **F1 = 2 × precision × recall / (precision + recall)**. It is high only
  when both precision and recall are reasonably high. It ranges from 0 to 1
  (higher is better).
- **Macro F1:** calculate an F1 score **separately for each land-cover class**,
  then take the simple average. Every class gets equal weight, even if some
  classes have fewer samples. We use this as our main comparison metric.
- **Confusion matrix:** a table of counts showing **actual/reference classes
  in rows** and **predicted classes in columns**. Counts on the diagonal are
  correct; counts off the diagonal are mistakes. It reveals *which* land-cover
  types the classifier confuses, not just how often it is right overall.

**Mini-example:** Of 10 true cotton pixels, suppose the model identifies 8 as
cotton and misses 2. Its cotton **recall is 8/10 = 80%**. If it predicts
“cotton” for 10 pixels altogether, and only 8 are truly cotton, its cotton
**precision is 8/10 = 80%** and its cotton **F1 is 0.80**. These are illustrative
numbers, **not results from this notebook**.

Use **macro F1** as the main score: calculate F1 for each class and average
across classes equally. Also inspect accuracy and the confusion matrix, since
one headline score can conceal poor predictions for a particular class.

The publisher supplies a separate final test file. Leave it unused until we
have chosen a method. This follows the source's official training/test split;
our validation subset is only for development within its training file.

**Limitation:** this is a random development holdout, not a spatial holdout.
Nearby or overlapping neighbourhoods may appear in different partitions.
Results cannot establish accuracy in a new geographic region.

In [ ]:
fit_idx, val_idx = train_test_split(
    np.arange(len(y)), test_size=.25, random_state=SEED, stratify=y)

# Simple verification: check the data and the separation before modelling.
assert data.shape == (4435, 37)
assert np.isfinite(data).all()
assert set(y) == set(CLASSES)
assert np.array_equal(data[:, 36], y)
assert X.min() >= 0 and X.max() <= 255
assert set(fit_idx).isdisjoint(val_idx)
assert set(y[fit_idx]) == set(y[val_idx]) == set(CLASSES)
assert not (set(map(tuple, X[fit_idx])) & set(map(tuple, X[val_idx])))
print('Checks passed. Fitting:', len(fit_idx), 'Validation:', len(val_idx))

## 4. Build a baseline we can explain

A Random Forest combines many decision trees. Each tree learns partitions of
the feature space; the forest combines their class probabilities to choose a
prediction. It can capture nonlinear relationships and does not usually need
standardized inputs.

Start with the central pixel's four bands. Compare it with a model that always
predicts the most common fitting class. If our classifier cannot improve on
that simple reference, we should investigate before making a scientific claim.

The short function below fits only on the fitting observations and evaluates
only on validation observations. It keeps the fitted model and predictions
in memory so we can inspect them in this notebook. `center` uses four bands;
`patch` adds the neighbouring pixels. No external runner is involved.

In [ ]:
experiments = {}

def get_features(mode, measurements=None):
    measurements = X if measurements is None else measurements
    if mode == 'center':
        return measurements[:, 16:20]
    if mode == 'patch':
        return measurements
    raise ValueError('Choose center or patch, or justify a new feature function.')

def run_experiment(name, model, features='center', reason=''):
    if name in experiments:
        raise ValueError('Use a new name so the earlier result is retained.')
    inputs = get_features(features)
    fitted = clone(model).fit(inputs[fit_idx], y[fit_idx])
    pred = fitted.predict(inputs[val_idx])
    experiments[name] = dict(
        model=fitted, features=features, prediction=pred, reason=reason,
        macro_f1=f1_score(y[val_idx], pred, labels=CLASSES, average='macro', zero_division=0),
        accuracy=accuracy_score(y[val_idx], pred))
    print(name, 'validation macro F1:', round(experiments[name]['macro_f1'], 3))

run_experiment('Majority class', DummyClassifier(strategy='most_frequent'),
               reason='Does a learned model beat the simplest prediction?')
run_experiment('Baseline forest', RandomForestClassifier(
    n_estimators=100, min_samples_leaf=2, random_state=SEED, n_jobs=1),
    reason='Baseline using only the central pixel bands.')

## 5. Look at the mistakes before changing the method

Rows of the confusion matrix are reference classes; columns are predicted
classes. Off-diagonal values show mistakes. Which class pair is most confused?
Could a neighbouring pixel help distinguish them, or might the label itself
be uncertain? Give an explanation and suggest an experiment to test it.

In [ ]:
baseline = experiments['Baseline forest']
print(classification_report(y[val_idx], baseline['prediction'], labels=CLASSES,
                            target_names=CLASS_NAMES, zero_division=0))
fig, ax = plt.subplots(figsize=(7, 6), layout='constrained')
ConfusionMatrixDisplay.from_predictions(
    y[val_idx], baseline['prediction'], labels=CLASSES,
    display_labels=CLASSES, ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Baseline forest — validation errors by class code')
plt.show()

## 6. Try one alternative ourselves

**Hypothesis:** a simpler model might already explain much of the spectral
separation. Try logistic regression with standardized bands.

Scaling is learned inside a pipeline, using the fitting observations only.
Do not scale the whole dataset before splitting: that would let information
from validation affect preprocessing. This compares whole pipelines; it does
not isolate the effect of scaling from the change in classifier.

In [ ]:
run_experiment('Scaled logistic regression', make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    reason='Can a linear model with scaled central bands compete with the forest?')

## 7. Add two understandable checks

**Verification:** refitting the baseline with the same inputs and seed should
reproduce its predictions. This checks repeatability, not scientific validity.

**Validation control:** shuffle the fitting labels and retrain. This destroys
the intended input–target relationship. Its validation score should fall
substantially. If it does not, investigate leakage or the evaluation before
trying more sophisticated models. Do not treat a particular chance F1 value
as universal; it depends on the class distribution and predictions.

In [ ]:
central = get_features('center')
repeat = clone(baseline['model']).fit(central[fit_idx], y[fit_idx])
assert np.array_equal(repeat.predict(central[val_idx]), baseline['prediction'])
print('Repeatability check passed.')

rng = np.random.default_rng(SEED)
shuffled = clone(baseline['model']).fit(central[fit_idx], rng.permutation(y[fit_idx]))
shuffled_f1 = f1_score(y[val_idx], shuffled.predict(central[val_idx]),
                      labels=CLASSES, average='macro', zero_division=0)
print('Real labels macro F1:', round(baseline['macro_f1'], 3))
print('Shuffled fitting labels macro F1:', round(shuffled_f1, 3))

## 8. Hand the method investigation to a coding agent

We have established the question, data, baseline and evaluation. Now let the
AI choose what to try. The human controls what counts as evidence; the agent
can investigate how to improve the predictions.

Give your existing coding assistant this prompt:

> Work in this notebook. Our question is: which method best predicts pixel land
> cover from these satellite measurements? Read the baseline, confusion matrix
> and existing results. Propose and run up to five additional experiments,
> one at a time, using the validation observations for feedback.
>
> You may change preprocessing, feature representation, classifier and its
> parameters. For example, compare central bands with the full neighbourhood,
> try scaling or PCA inside a pipeline, or try kNN, SVM or another tree model.
> Choose the next experiment from the evidence rather than writing a large
> grid search. Include at least one comparison changing only a single choice,
> so we can understand what helped. Make new cells and retain earlier results.
>
> Keep the raw measurements, reference labels, fit_idx, val_idx, seed and macro
> F1 metric fixed. Fit preprocessing only on fitting observations. Do not
> download or examine sat.tst, run the final-test cell, or use class labels as
> predictors. Do not change labels to make a method score better.
>
> Before each experiment, state the hypothesis and why it is worth testing.
> Record the exact method, actual validation result, and whether the evidence
> supports the hypothesis. Keep failed attempts and negative results visible.
> Add a sensible check for your changes and explain what it establishes.
> Recommend a method and discuss class errors, limitations and what you would
> test next. If you cannot execute the code, say so; do not invent results.

**Observe the agent:** Did it use feedback to make a decision? Did its checks
test something useful? Can you explain the code and defend its conclusion?
The five-experiment limit is a classroom instruction, not software enforcement.

## 9. Compare the evidence

Re-run this cell after the agent finishes. These are validation scores used to
choose methods, so repeated searching can overfit them. “Best” means best among
the methods actually tried, on this dataset under this evaluation.

For your presentation, use the method comparison and a confusion matrix.
Explain which changes helped and which did not. Do not make a geographic map
from the shuffled table: coordinates are unavailable.

In [ ]:
results = pd.DataFrame([
    dict(method=name, macro_f1=result['macro_f1'], accuracy=result['accuracy'],
         features=result['features'], reason=result['reason'])
    for name, result in experiments.items()])
display(results.sort_values('macro_f1', ascending=False))
fig, ax = plt.subplots(figsize=(9, 4), layout='constrained')
ax.barh(results['method'], results['macro_f1'])
ax.set(xlim=(0, 1), xlabel='Validation macro F1', title='Which tested method predicts best?')
plt.show()

## 10. One final test after choosing a method

Once the search is finished, choose the highest validation macro F1. Freeze
that choice, refit it on all development observations, and compare with the
baseline on the publisher's separate test set. Use that set once, not to choose
the next experiment. Do not change the selected method because another scores
better on this final test.

The switch below defaults to False so **Run All does not download or use the final test**.
The human sets it to True after reviewing the investigation. Rerunning the
notebook does not make a previously viewed test set unseen again.

In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    chosen_name = max(experiments, key=lambda name: experiments[name]['macro_f1'])
    chosen = deepcopy(experiments[chosen_name])
    print('Frozen validation choice:', chosen_name)
    test_path = download_file('sat.tst')
    test_data = np.loadtxt(test_path)
    assert test_data.shape == (2000, 37) and np.isfinite(test_data).all()
    test_X, test_y = test_data[:, :36], test_data[:, 36].astype(int)
    assert set(test_y) == set(CLASSES)
    final_rows = []
    for name in dict.fromkeys(['Baseline forest', chosen_name]):
        result = experiments[name]
        train_features = get_features(result['features'])
        test_features = get_features(result['features'], test_X)
        final_model = clone(result['model']).fit(train_features, y)
        prediction = final_model.predict(test_features)
        final_rows.append(dict(method=name, test_macro_f1=f1_score(
            test_y, prediction, labels=CLASSES, average='macro', zero_division=0),
            test_accuracy=accuracy_score(test_y, prediction)))
    display(pd.DataFrame(final_rows))
else:
    print('Final test not run. Complete the investigation and choose a method first.')

## Discussion and a short research briefing

Prepare a five-minute explanation with 1–2 clear graphics:

1. What is measured, what is predicted, and where do the reference labels come from?
2. What baseline did we build and how did we check it?
3. What did the agent choose to change, and what happened?
4. Which method is supported by the evidence, and what remains uncertain?

We cannot claim geographic generalization from this table. A stronger next
study would use georeferenced satellite imagery, independently labelled
locations, and spatially separated fitting, validation and test regions.
Prediction maps could then show where the selected method succeeds and fails.

## Sources

- Srinivasan, A. (1993). *Statlog (Landsat Satellite)*. UCI Machine Learning
  Repository. https://doi.org/10.24432/C55887. Dataset licensed CC BY 4.0;
  downloaded `sat.doc` records its provenance and band layout.
- Edward Oughton, GGS416 *Satellite Image Analysis*,
  [Week 5 pixel classification](https://github.com/edwardoughton/satellite-image-analysis/blob/1cbe05d928445d31cdc424c03345a412436285af/05_01_ggs416_26_02_23.ipynb).
  Adapted teaching concepts: pixel feature vectors, classification, Random
  Forest baseline, confusion-matrix diagnosis and method interpretation.
  The source's threshold labels and injected-noise demonstration are not used
  as the reference labels or main experiment here.

